In [2]:
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
llm = fr"{project_root}\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\2024 NBKB 203_v1.3.html"
verified = fr"{project_root}\data\Documents_Annotés\2024 NBKB 203_LLMv1.3_Verified_EG.htmL"

llm = fr"{project_root}\data\Documents_Annotés\llm\TEST_p2_c500_fsselected-30_mgpt-5.2\2005QCCA437_v1.3.html"
verified = fr"{project_root}\data\final\Annotated\2005QCCA437_LLMv1.3_Verified_GL_revRL_tech.html"
with open(llm, "r", encoding="utf-8") as f:
    llm_html = f.read()
with open(verified, "r", encoding="utf-8") as f:
    verified_html = f.read()

In [3]:
import re
from typing import Dict, List, Tuple, Optional
from bs4 import BeautifulSoup, NavigableString, Tag

def extract_body(html_content: str) -> str:
    """
    Extract only the body content from HTML, excluding style, script, and head tags.
    Returns the exact string representation of the <body> element to keep reversibility.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    body = soup.find('body')
    if body is not None:
        return str(body)
    print("   ⚠ Warning: No <body> tag found, returning original content")
    return html_content
def tokenize(html_body: str, print=False) -> list:
    """
    Split HTML into a sequence of tokens that preserves:
    - HTML tags as single tokens (e.g., '<div class="x">')
    - Whitespace runs as separate tokens (spaces, newlines, tabs)
    - Punctuation as separate tokens (e.g., ',', '.', ';')
    - Words as separate tokens
    This ensures decode(tokens) == html_body with simple join AND prevents
    merging of words with punctuation after tag removal.
    """
    # Enhanced pattern: separates tags, whitespace, punctuation, words, and other chars
    # Group 1: HTML tags
    # Group 2: Whitespace runs
    # Group 3: Common punctuation (as separate tokens)
    # Group 4: Word characters (alphanumeric + underscore)
    # Group 5: Any other single character
    pattern = re.compile(r"(<[^>]*>)|(\s+)|([.,;:!?()[\]{}\"'`‑–—])|(\w+)|([^\w\s<>])")
    tokens = []
    for m in pattern.finditer(html_body):
        tok = m.group(1) or m.group(2) or m.group(3) or m.group(4) or m.group(5)
        if tok:  # Safety check
            tokens.append(tok)

    if print:
        print(f"   ✓ Tokenized into {len(tokens)} tokens (tags+whitespace+punctuation+words, reversible)")
    return tokens


def decode(tokens: list, print=False) -> str:
    """
    Reconstruct HTML body by concatenating tokens exactly.
    """
    reconstructed_html = "".join(tokens)

    if print:
        print(f"   ✓ Decoded {len(tokens)} tokens into HTML")
    return reconstructed_html


def is_tag(token: str) -> bool:
    return token.startswith("<") and token.endswith(">")


def is_opening_label(token: str) -> Tuple[bool, str]:
    """Returns (is_label, tag_type) where tag_type is 'manual_label' or 'auto_label'."""
    for tag_type in ("manual_label", "auto_label"):
        # Match <manual_label ...> or <manual_label> (with or without attributes)
        if re.match(rf"<{tag_type}(\s|>)", token):
            return True, tag_type
    return False, ""


def is_closing_label(token: str, tag_type: str) -> bool:
    return token == f"</{tag_type}>"


def parse_label_attributes(token: str) -> Tuple[str, Dict[str, str]]:
    """
    Extract labelname and other attributes from an opening label tag token.
    e.g. '<manual_label labelname="FOO" color="red">'
    """
    labelname = ""
    attributes = {}

    for m in re.finditer(r'(\w+)=["\']([^"\']*)["\']', token):
        key, val = m.group(1), m.group(2)
        if key == "labelname":
            labelname = val
        elif key != "style":
            attributes[key] = val

    return labelname, attributes

In [4]:
from bs4 import BeautifulSoup
from collections import defaultdict
from typing import Dict, List, Tuple
import os
from pathlib import Path
import re
from datetime import datetime



from typing import Dict, List, Tuple

class Span:
    def __init__(self, text: str, start: int, end: int, labelname: str,
                 attributes: Dict[str, str], context_text: str = "", type: str = "manual_label"):
        self.text = text.strip()
        self.start = start
        self.end = end
        self.labelname = labelname
        self.attributes = attributes
        self.context_text = context_text
        self.type = type

    def __repr__(self):
        return f"Span('{self.text[:30]}...', label={self.labelname}, start={self.start})"


def extract_spans_from_html(html_body: str, context_chars: int = 200) -> Tuple[str, List["Span"]]:
    """
    Tokenize the HTML body once, then walk token by token:
    - Tag tokens are skipped when building plain text (they don't contribute characters)
    - Non-tag tokens advance the plain-text offset exactly
    - When an opening label tag is encountered, we snapshot the current offset
    - When the matching closing tag is encountered, we know start and end precisely
    - Context is a slice of the plain text around [start, end] — no searching at all

    Handles nesting: a label inside another label works correctly because
    we track a stack. The outer label's end is recorded when *its* closing
    tag is seen, by which point the inner label has already been fully processed.
    """
    tokens = tokenize(html_body)

    # --- Pass 1: build the plain text string and a token→char-offset map ---
    # plain_offset[i] = the plain-text offset BEFORE token i is consumed.
    # For tag tokens the offset does not advance (they contribute 0 chars).
    plain_chars: List[str] = []
    token_start_offset: List[int] = []  # plain-text offset at the START of each token

    for tok in tokens:
        token_start_offset.append(len(plain_chars))
        if not is_tag(tok):
            plain_chars.extend(tok)  # extend char by char to keep offsets exact

    plain_text = "".join(plain_chars)
    # Sentinel: offset after the last token
    token_start_offset.append(len(plain_text))

    # --- Pass 2: walk tokens, detect labels, emit Spans ---
    spans: List["Span"] = []

    # Stack entries: (tag_type, labelname, attributes, plain_text_start, token_index_of_open)
    stack: List[Tuple[str, str, Dict, int]] = []

    for i, tok in enumerate(tokens):
        is_lbl, tag_type = is_opening_label(tok)

        if is_lbl:
            labelname, attributes = parse_label_attributes(tok)
            plain_start = token_start_offset[i]  # offset right before this tag
            stack.append((tag_type, labelname, attributes, plain_start))

        elif is_tag(tok) and stack:
            # Check if this closes the innermost open label
            top_tag_type = stack[-1][0]
            if is_closing_label(tok, top_tag_type):
                tag_type, labelname, attributes, plain_start = stack.pop()
                # plain-text offset right after this closing tag = same as start of next token
                plain_end = token_start_offset[i]  # closing tag itself adds 0 chars

                span_text = plain_text[plain_start:plain_end]
                normalized_text = " ".join(span_text.split())

                if normalized_text:
                    ctx_start = max(0, plain_start - context_chars)
                    ctx_end = min(len(plain_text), plain_end + context_chars)
                    context_text = plain_text[ctx_start:ctx_end]

                    spans.append(Span(
                        text=normalized_text,
                        start=plain_start,
                        end=plain_end,
                        labelname=labelname,
                        attributes=attributes,
                        context_text=context_text,
                        type=tag_type,
                    ))

    return plain_text, spans

In [20]:
import re


def jaccard_context_overlap(span1, span2, threshold: float = 0.7) -> bool:
    """
    Check approximate overlap between two spans using Jaccard similarity
    over the union of their context_text and text words (case-insensitive).
    """
    ctx1 = f"{span1.context_text} {span1.text}".lower() if span1.context_text is not None else span1.text.lower()
    ctx2 = f"{span2.context_text} {span2.text}".lower() if span2.context_text is not None else span2.text.lower()
    
    words1 = set(ctx1.split())
    words2 = set(ctx2.split())
    
    if not words1 or not words2:
        return False
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    if union == 0:
        return False
    
    similarity = intersection / union
    return similarity >= threshold

def share_text_word(span1, span2) -> bool:
    """
    Return True if the two spans share at least one word in their *text* fields.
    Used to define overlap once context is approximately the same.
    Now robust to trailing punctuation like commas/periods (e.g. 'ewert' vs 'ewert,').
    """
    # Normalize by stripping punctuation and extracting word characters only
    words1 = set(re.findall(r"\w+", span1.text.lower()))
    words2 = set(re.findall(r"\w+", span2.text.lower()))
    
    if not words1 or not words2:
        return False
    
    return len(words1 & words2) > 0

In [5]:
_, spans1 = extract_spans_from_html(html_body=extract_body(llm_html), context_chars=200)
_, spans2 = extract_spans_from_html(html_body=extract_body(verified_html), context_chars=200)
print(f"\n✓ LLM: {len(spans1)} spans")
print(f"✓ Verified: {len(spans2)} spans")


✓ LLM: 316 spans
✓ Verified: 301 spans


In [6]:
s2_manual_label = [s for s in spans2 if s.type == "manual_label"]
s2_auto_label = [s for s in spans2 if s.type == "auto_label"]
print(f"\n✓ Verified: {len(s2_manual_label)} manual-labeled spans")
print(f"✓ Verified: {len(s2_auto_label)} auto-labeled spans")


✓ Verified: 72 manual-labeled spans
✓ Verified: 229 auto-labeled spans


In [22]:
matched = []
matched_idx = []
for i, s1 in enumerate(spans1):

    for j, s2 in enumerate(spans2):

        if j in [k for _, k in matched_idx]:
            continue

        if s1.type != s2.type:
            continue

        if s1.labelname != s2.labelname:
            continue

        if s1.text != s2.text:
            continue

        if not jaccard_context_overlap(s1, s2, threshold= 0.9):
            continue

        matched.append((s1, s2))
        matched_idx.append((i, j))

        break

print(f"\n✓ Exact matches (same text, same label type): {len(matched)} spans")


✓ Exact matches (same text, same label type): 229 spans


In [23]:
non_matched_auto_spans = [s1 for i, s1 in enumerate(spans1) if i not in [k for k, _ in matched_idx]]

In [24]:
for s1 in non_matched_auto_spans:
    print("====================")
    print(s1.text)
    print("----------")
    print(s1.context_text)
    print("====================\n\n\n\n")

Pharmascience inc. c. Option Consommateurs
----------


 



Pharmascience
  inc. c. Option Consommateurs


 2005
  QCCA 437
 



COURT OF
  APPEAL




CANADA




PROVINCE
  OF QUEBEC




MONTRÉAL REGISTRY


 




 




No:


500-09-014659-049




 


(500-06-000192-035)




 




DATE:


April 29, 2005





2005 QCCA 437
----------


 



Pharmascience
  inc. c. Option Consommateurs


 2005
  QCCA 437
 



COURT OF
  APPEAL




CANADA




PROVINCE
  OF QUEBEC




MONTRÉAL REGISTRY


 




 




No:


500-09-014659-049




 


(500-06-000192-035)




 




DATE:


April 29, 2005




_______________




articles 1002
----------
 reasons of Justice Paul-Arthur Gendreau,
with which Chief Justice Michel Robert and Justice André Rochon agree;
[4]               
DISMISSES the
appeal, with costs;
[5]               
DECLARES valid
articles 1002 and 1003 C.C.P.;
[6]               
DISMISSES the
appellant’s two motions of April 19, 2004, brought before the Superior Court.



 




 



_________

In [26]:
print(len(non_matched_auto_spans))

87


In [27]:
removed_auto_no_manual = []
auto_with_manual_candidate = []

for n_s1 in non_matched_auto_spans:
    found_overlap = False

    for s2 in s2_manual_label:
        
        
        # 1) same approximate context
        if not jaccard_context_overlap(n_s1, s2, threshold=0.9):
            continue
        
        # 2) at least one common word in span.text
        if share_text_word(n_s1, s2):
            auto_with_manual_candidate.append((n_s1, s2))
            found_overlap = True
            break
        
    
    # If we never found a manual span with same approx context AND
    # at least one shared word in the text, then this auto is truly removed.
    if not found_overlap:
        #print("--------------------------------------------------------")
        #print("not matched : truly removed")
        #print(n_s1.text, n_s1.labelname, n_s1.type)
        #print(n_s1.context_text)
        removed_auto_no_manual.append(n_s1)


print(f"Non-matched auto spans: {len(non_matched_auto_spans)}")
print(f"→ With likely manual replacement (overlap): {len(auto_with_manual_candidate)}")
print(f"→ Truly removed (no overlap under same approx context): {len(removed_auto_no_manual)}")

print("\nExamples of truly removed auto spans:")
for span in removed_auto_no_manual:
    print("-", span.text[:120].replace("\n", " "), f"[label={span.labelname}, type={span.type}]")

Non-matched auto spans: 87
→ With likely manual replacement (overlap): 58
→ Truly removed (no overlap under same approx context): 29

Examples of truly removed auto spans:
- Pharmascience inc. c. Option Consommateurs [label=title, type=auto_label]
- 2005 QCCA 437 [label=citation, type=auto_label]
- Justice Carole Julien of the Superior Court [label=decision, type=auto_label]
- Quebec [label=title, type=auto_label]
- the act [label=title, type=auto_label]
- the act [label=legislation, type=auto_label]
- regulation [label=title, type=auto_label]
- regulation [label=legislation, type=auto_label]
- regulation [label=fragment, type=auto_label]
- regulation [label=legislation, type=auto_label]
- Minister of Health [label=legislation, type=auto_label]
- Regulation [label=title, type=auto_label]
- Regulation [label=legislation, type=auto_label]
- [60] [label=legislation, type=auto_label]
- André Noël [label=authors, type=auto_label]
- Des millions en primes illégales verses aux pharmaciens [la

In [28]:
for s2 in s2_manual_label:
    print("====================")
    print(s2.text)
    print("----------")
    print(s2.context_text)
    print("====================\n\n\n\n")

articles 1002
----------
 reasons of Justice Paul-Arthur Gendreau,
with which Chief Justice Michel Robert and Justice André Rochon agree;
[4]               
DISMISSES the
appeal, with costs;
[5]               
DECLARES valid
articles 1002 and 1003 C.C.P.;
[6]               
DISMISSES the
appellant’s two motions of April 19, 2004, brought before the Superior Court.



 




 



__________________________________

J.J. MICHEL ROBERT C.




C.C.P.
----------
eurs, Pharmascience and the other defendant pharmaceutical
corporations raised a number of preliminary exceptions, including one seeking a
declaration of the constitutional invalidity of article 1002 C.C.P. They
claimed that a recent amendment to this provision deprived them of their right
to a full defence at the authorization stage of the action. Justice Carole
Julien of the Superior Court dismissed t




article 1002 C.C.P.
----------
ion
Consommateurs, Pharmascience and the other defendant pharmaceutical
corporations raised a num

In [29]:
count=0
for spans in removed_auto_no_manual:
    for s2 in s2_manual_label:
        
        
        # 1) same approximate context
        if not jaccard_context_overlap(spans, s2, threshold=0.9):
            continue
        

        # 2) at least one common word in span.text
        words1 = set(spans.text.lower().split())
        words2 = set(s2.text.lower().split())

        print(words1)
        print(words2)
        print(len(words1 & words2))

        if share_text_word(spans, s2):
            auto_with_manual_candidate.append((spans, s2))
            found_overlap = True
            break

In [30]:
print(len(auto_with_manual_candidate))

58


In [31]:
# Build a clear, formatted summary report of the verification process
print("\n================ Annotation Verification Report ================")

# Compute derived sets if needed
s2_manual_label = [s for s in spans2 if s.type == "manual_label"]
s2_auto_label = [s for s in spans2 if s.type == "auto_label"]
total_verified_spans = len(spans2) 

modified_auto = len(auto_with_manual_candidate)

new_manual = len(s2_manual_label) - modified_auto
removed_auto = len(removed_auto_no_manual)

# Percentages relative to the verified document
def pct(part, whole):
    return (part / whole * 100) if whole else 0
pct_modified_auto = pct(modified_auto, total_verified_spans)
pct_new_manual = pct(new_manual, total_verified_spans)
pct_removed_auto = pct(removed_auto, total_verified_spans)

report = f"""
First document (entirely annotated by an LLM) has {len(spans1)} auto-labeled spans,
while the verified version has {len(spans2)} labeled spans.

In the verified document there are {len(s2_manual_label)} manual-labeled spans
and {len(s2_auto_label)} auto-labeled spans.

There are {modified_auto} modified auto_label spans
({pct_modified_auto:.1f}% of spans in the verified document).

Hence there are {new_manual} new manual-labeled spans that were not in the original LLM annotation
({pct_new_manual:.1f}% of spans in the verified document).

Hence there are {removed_auto} auto-labeled spans that were removed during the verification process
({pct_removed_auto:.1f}% of spans in the verified document).
"""

print(report)
print("===============================================================\n")


================ Annotation Verification Report ================

First document (entirely annotated by an LLM) has 316 auto-labeled spans,
while the verified version has 301 labeled spans.

In the verified document there are 72 manual-labeled spans
and 229 auto-labeled spans.

There are 58 modified auto_label spans
(19.3% of spans in the verified document).

Hence there are 14 new manual-labeled spans that were not in the original LLM annotation
(4.7% of spans in the verified document).

Hence there are 29 auto-labeled spans that were removed during the verification process
(9.6% of spans in the verified document).




In [32]:
for s1, s2 in auto_with_manual_candidate:
    print("====================")
    print(s1.text)
    print(s2.text)

articles 1002
articles 1002
article 1002 C.C.P.
C.C.P.
article 1002
article 1002
C.C.P.
C.C.P.
article 1002 C.C.P.
article 1002
C.C.P.
C.C.P.
article 1002 C.C.P.
C.C.P.
C.C.P.
C.C.P.
article 1002 C.C.P.
C.C.P.
article 1003 C.C.P.
C.C.P.
C.C.P.
C.C.P.
article 1002 C.C.P.
C.C.P.
article 1002 C.C.P.
C.C.P.
article 1002 C.C.P.
C.C.P.
Code is
Code
Code is
Code
C.C.P.
C.C.P.
article 1003 C.C.P.
C.C.P.
article 93 C.C.P.
C.C.P.
C.C.P.
C.C.P.
1003 C.C.P.
C.C.P.
Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of
Crane Canada Inc. v. Sécurité nationale, compagnie d’assurances and Attorney General of Quebec
Article 13
Article 13
Code of Civil Procedure
Code of Civil Procedure
Article 13 of the Code of Civil Procedure
Article 13
Article 396.1
Article 396.1
C.C.P.
C.C.P.
Article 396.1 C.C.P.
Article 396.1
Quebec’s Act respecting the class action
Act respecting the class action
André Noël, “Des millions en primes illégales verses aux pharmaciens,” La Presse (Febru

In [34]:
only_in_s1 = [s1 for s1, _ in auto_with_manual_candidate]
only_in_s2 = [s2 for _, s2 in auto_with_manual_candidate]

print(f"mean length of spans in the llm document (only for the modified spans): {sum(len(s.text) for s in only_in_s1)/len(only_in_s1)}")
print(f"mean length of spans in the verified document (only for the modified spans): {sum(len(s.text) for s in only_in_s2)/len(only_in_s2)}")

mean length of spans in the llm document (only for the modified spans): 22.46551724137931
mean length of spans in the verified document (only for the modified spans): 21.344827586206897


In [35]:
import difflib
from collections import Counter

def analyze_edge_modifications(s1_text: str, s2_text: str):
    """
    Use the existing `tokenize` function to compare two span texts token-by-token
    and characterize edge changes going from s1 → s2 as:
      - extend_left / reduce_left / same / complex_left
      - extend_right / reduce_right / same / complex_right
    """
    # Tokenize and drop pure-whitespace tokens (spaces/newlines are not informative here)
    t1 = [t for t in tokenize(s1_text) if not t.isspace()]
    t2 = [t for t in tokenize(s2_text) if not t.isspace()]
    
    sm = difflib.SequenceMatcher(a=t1, b=t2)
    blocks = sm.get_matching_blocks()
    if len(blocks) == 0:
        return {
            "left_change": "no_common_subsequence",
            "right_change": "no_common_subsequence",
            "left_removed": t1,
            "left_added": t2,
            "right_removed": [],
            "right_added": [],
        }
    
    # First and last *real* blocks (last element is a 0-length sentinel)
    first = blocks[0]
    last = blocks[-2] if len(blocks) > 1 else blocks[0]
    
    # Left side analysis
    # first.a = index in t1 where the first common block starts
    # first.b = index in t2 where the first common block starts
    if first.a == 0 and first.b == 0:
        left_change = "same"
    elif first.a > 0 and first.b == 0:
        # Extra tokens at the left in s1 that disappeared in s2
        left_change = "reduce_left"
    elif first.a == 0 and first.b > 0:
        # Extra tokens at the left in s2 that were not in s1
        left_change = "extend_left"
    else:
        # Both sides have unmatched prefixes → more complex than pure extend/reduce
        left_change = "complex_left"
    
    left_removed = t1[:first.a] if first.a > 0 else []
    left_added = t2[:first.b] if first.b > 0 else []
    
    # Right side analysis
    end1 = len(t1)
    end2 = len(t2)
    a_end = last.a + last.size
    b_end = last.b + last.size
    
    if a_end == end1 and b_end == end2:
        right_change = "same"
    elif a_end < end1 and b_end == end2:
        # Extra tokens at the right in s1 that disappeared in s2
        right_change = "reduce_right"
    elif a_end == end1 and b_end < end2:
        # Extra tokens at the right in s2 that were not in s1
        right_change = "extend_right"
    else:
        right_change = "complex_right"
    
    right_removed = t1[a_end:] if a_end < end1 else []
    right_added = t2[b_end:] if b_end < end2 else []
    
    return {
        "left_change": left_change,
        "right_change": right_change,
        "left_removed": left_removed,
        "left_added": left_added,
        "right_removed": right_removed,
        "right_added": right_added,
    }

# Apply to all (s1, s2) pairs in auto_with_manual_candidate
edge_change_counts = Counter()

for idx, (s1, s2) in enumerate(zip(only_in_s1, only_in_s2), start=1):
    res = analyze_edge_modifications(s1.text, s2.text)
    key = (res["left_change"], res["right_change"])
    edge_change_counts[key] += 1
    
    # Print a few illustrative examples
    if idx <= 96:
        print("====================")
        print(f"Pair {idx}")
        print("s1:", s1.text)
        print("s2:", s2.text)
        print("Texts equal:", s1.text == s2.text)
        print("Left change:", res["left_change"],"| removed:", res["left_removed"],"| added:", res["left_added"])
        print("Right change:", res["right_change"],"| removed:", res["right_removed"],"| added:", res["right_added"])

print("\n===== Summary of edge modifications (s1 → s2) =====")
for (left, right), count in edge_change_counts.items():
    print(f"Left={left:14s} | Right={right:14s} : {count}")

Pair 1
s1: articles 1002
s2: articles 1002
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 2
s1: article 1002 C.C.P.
s2: C.C.P.
Texts equal: False
Left change: reduce_left | removed: ['article', '1002'] | added: []
Right change: same | removed: [] | added: []
Pair 3
s1: article 1002
s2: article 1002
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 4
s1: C.C.P.
s2: C.C.P.
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 5
s1: article 1002 C.C.P.
s2: article 1002
Texts equal: False
Left change: same | removed: [] | added: []
Right change: reduce_right | removed: ['C', '.', 'C', '.', 'P', '.'] | added: []
Pair 6
s1: C.C.P.
s2: C.C.P.
Texts equal: True
Left change: same | removed: [] | added: []
Right change: same | removed: [] | added: []
Pair 7
s1: article 1002 C.C.P.
s2: C.C.P.
Texts equal: False
Lef

Left=same           | Right=same           : 43

Left=reduce_left    | Right=same           : 9

Left=same           | Right=reduce_right   : 7

Left=same           | Right=extend_right   : 32

Left=extend_left    | Right=same           : 5